# Create Healthcare System Ontology via Fabric REST API

This notebook programmatically creates a **Healthcare System Ontology** in Microsoft Fabric using the Ontology REST API. It builds the full ontology definition — entity types, data bindings, relationship types, and contextualizations — then deploys it to your workspace in a single API call.

**Prerequisites:**
- A Microsoft Fabric workspace with Contributor (or higher) permissions
- A Lakehouse containing five tables: `patient`, `provider`, `appointment`, `diagnosis`, `prescription`
- The Lakehouse must be attached to this notebook (used to resolve workspace and item IDs)


## 1. Install Dependencies

In [ ]:
%pip install semantic-link-labs

## 2. Imports

In [ ]:
import sempy_labs as labs
import sempy.fabric as fabric
import requests
import json
import base64
import uuid
import time

## 3. Parameters

Set your workspace and lakehouse names below. If you leave them as `None`, the notebook will use the currently attached workspace/lakehouse.

In [ ]:
# ─── User Parameters ───────────────────────────────────────────────────────────
# Set these to match your environment, or leave as None for the current context.

workspace_name = None          # e.g. "My Healthcare Workspace"
lakehouse_name = None          # e.g. "HealthcareLakehouse"
ontology_display_name = "HealthcareSystemOntology"
ontology_description = "Patient care management with providers, appointments, and treatments"


## 4. Resolve Workspace and Lakehouse IDs

We need the workspace ID and lakehouse item ID to build data binding references.

In [ ]:
# Resolve workspace ID
workspace_id = fabric.resolve_workspace_id(workspace_name)
print(f"Workspace ID: {workspace_id}")

# Resolve lakehouse ID
lakehouses_df = fabric.list_items(workspace=workspace_name, type="Lakehouse")

if lakehouse_name:
    lh_row = lakehouses_df[lakehouses_df["Display Name"] == lakehouse_name]
else:
    # Use the first lakehouse if none specified
    lh_row = lakehouses_df.head(1)

if lh_row.empty:
    raise ValueError("Lakehouse not found. Check lakehouse_name parameter or attach a lakehouse to this notebook.")

lakehouse_id = str(lh_row.iloc[0]["Id"])
lakehouse_display = lh_row.iloc[0]["Display Name"]
print(f"Lakehouse: {lakehouse_display} ({lakehouse_id})")


## 5. Build the Ontology Definition

This cell constructs the full ontology definition payload: 5 entity types with data bindings, and 6 relationship types with contextualizations. Each JSON structure is base64-encoded per the Fabric Ontology REST API specification.

**Entity Types:** Patient, Provider, Appointment, Diagnosis, Prescription  
**Relationship Types:** has_appointment, sees, diagnosed_with, diagnoses, treated_by, prescribes

In [ ]:
# ─── Helper: encode a dict to base64 ──────────────────────────────────────────
def to_b64(obj):
    """Convert a Python dict to a base64-encoded JSON string (Fabric API format)."""
    json_str = json.dumps(obj, indent=2)
    return base64.b64encode(json_str.encode("utf-8")).decode("utf-8")


# ─── Helper: generate unique 64-bit integer IDs ──────────────────────────────
# Fabric ontology IDs are positive 64-bit integers. We use a counter starting
# at a high value to avoid collisions.
_id_counter = 1000000000000

def next_id():
    global _id_counter
    _id_counter += 1
    return str(_id_counter)


# ─── Entity Type IDs ─────────────────────────────────────────────────────────
et_ids = {
    "Patient":      next_id(),
    "Provider":     next_id(),
    "Appointment":  next_id(),
    "Diagnosis":    next_id(),
    "Prescription": next_id(),
}

# ─── Property definitions per entity type ────────────────────────────────────
# Each property gets a unique ID. We track them in a dict for cross-referencing
# in data bindings and relationships.

prop_ids = {}  # key: "EntityName.propertyName" -> id string

def make_property(entity, name, value_type="String"):
    pid = next_id()
    prop_ids[f"{entity}.{name}"] = pid
    return {
        "id": pid,
        "name": name,
        "redefines": None,
        "baseTypeNamespaceType": None,
        "valueType": value_type,
    }


# ─── VALUE TYPE MAPPING ──────────────────────────────────────────────────────
# Maps the RDF property types to Fabric ontology value types
rdf_to_fabric_type = {
    "string": "String",
    "integer": "BigInt",
    "date": "DateTime",
    "datetime": "DateTime",
}

# ─── ENTITY TYPE DEFINITIONS ─────────────────────────────────────────────────

entity_definitions = {}

# --- Patient ---
patient_props = [
    make_property("Patient", "patientId", "String"),
    make_property("Patient", "name", "String"),
    make_property("Patient", "mrn", "String"),
    make_property("Patient", "dateOfBirth", "DateTime"),
    make_property("Patient", "bloodType", "String"),
    make_property("Patient", "allergies", "String"),
]
entity_definitions["Patient"] = {
    "id": et_ids["Patient"],
    "namespace": "usertypes",
    "baseEntityTypeId": None,
    "name": "Patient",
    "entityIdParts": [prop_ids["Patient.patientId"]],
    "displayNamePropertyId": prop_ids["Patient.name"],
    "namespaceType": "Custom",
    "visibility": "Visible",
    "properties": patient_props,
    "timeseriesProperties": [],
}

# --- Provider ---
provider_props = [
    make_property("Provider", "providerId", "String"),
    make_property("Provider", "name", "String"),
    make_property("Provider", "specialty", "String"),
    make_property("Provider", "licenseNumber", "String"),
    make_property("Provider", "department", "String"),
]
entity_definitions["Provider"] = {
    "id": et_ids["Provider"],
    "namespace": "usertypes",
    "baseEntityTypeId": None,
    "name": "Provider",
    "entityIdParts": [prop_ids["Provider.providerId"]],
    "displayNamePropertyId": prop_ids["Provider.name"],
    "namespaceType": "Custom",
    "visibility": "Visible",
    "properties": provider_props,
    "timeseriesProperties": [],
}

# --- Appointment ---
appointment_props = [
    make_property("Appointment", "appointmentId", "String"),
    make_property("Appointment", "patientId", "String"),
    make_property("Appointment", "providerId", "String"),
    make_property("Appointment", "scheduledTime", "DateTime"),
    make_property("Appointment", "duration", "BigInt"),
    make_property("Appointment", "type", "String"),
    make_property("Appointment", "status", "String"),
]
entity_definitions["Appointment"] = {
    "id": et_ids["Appointment"],
    "namespace": "usertypes",
    "baseEntityTypeId": None,
    "name": "Appointment",
    "entityIdParts": [prop_ids["Appointment.appointmentId"]],
    "displayNamePropertyId": prop_ids["Appointment.appointmentId"],
    "namespaceType": "Custom",
    "visibility": "Visible",
    "properties": appointment_props,
    "timeseriesProperties": [],
}

# --- Diagnosis ---
diagnosis_props = [
    make_property("Diagnosis", "diagnosisId", "String"),
    make_property("Diagnosis", "patientId", "String"),
    make_property("Diagnosis", "providerId", "String"),
    make_property("Diagnosis", "icdCode", "String"),
    make_property("Diagnosis", "description", "String"),
    make_property("Diagnosis", "severity", "String"),
    make_property("Diagnosis", "diagnosedDate", "DateTime"),
]
entity_definitions["Diagnosis"] = {
    "id": et_ids["Diagnosis"],
    "namespace": "usertypes",
    "baseEntityTypeId": None,
    "name": "Diagnosis",
    "entityIdParts": [prop_ids["Diagnosis.diagnosisId"]],
    "displayNamePropertyId": prop_ids["Diagnosis.description"],
    "namespaceType": "Custom",
    "visibility": "Visible",
    "properties": diagnosis_props,
    "timeseriesProperties": [],
}

# --- Prescription ---
prescription_props = [
    make_property("Prescription", "rxNumber", "String"),
    make_property("Prescription", "diagnosisId", "String"),
    make_property("Prescription", "providerId", "String"),
    make_property("Prescription", "medication", "String"),
    make_property("Prescription", "dosage", "String"),
    make_property("Prescription", "frequency", "String"),
    make_property("Prescription", "refillsRemaining", "BigInt"),
]
entity_definitions["Prescription"] = {
    "id": et_ids["Prescription"],
    "namespace": "usertypes",
    "baseEntityTypeId": None,
    "name": "Prescription",
    "entityIdParts": [prop_ids["Prescription.rxNumber"]],
    "displayNamePropertyId": prop_ids["Prescription.medication"],
    "namespaceType": "Custom",
    "visibility": "Visible",
    "properties": prescription_props,
    "timeseriesProperties": [],
}

print("Entity type definitions built successfully.")
for name, defn in entity_definitions.items():
    print(f"  {name}: ID={defn['id']}, {len(defn['properties'])} properties, key={defn['entityIdParts']}")


## 6. Build Data Bindings

Each entity type gets a `NonTimeSeries` data binding that maps its Lakehouse table columns to the ontology properties.

In [ ]:
# ─── DATA BINDINGS ────────────────────────────────────────────────────────────
# Maps: entity name -> (source table name, list of (column_name, entity.property_key))

binding_map = {
    "Patient": ("patient", [
        ("patientId", "Patient.patientId"),
        ("name",      "Patient.name"),
        ("mrn",       "Patient.mrn"),
        ("dateOfBirth","Patient.dateOfBirth"),
        ("bloodType", "Patient.bloodType"),
        ("allergies", "Patient.allergies"),
    ]),
    "Provider": ("provider", [
        ("providerId",   "Provider.providerId"),
        ("name",         "Provider.name"),
        ("specialty",    "Provider.specialty"),
        ("licenseNumber","Provider.licenseNumber"),
        ("department",   "Provider.department"),
    ]),
    "Appointment": ("appointment", [
        ("appointmentId","Appointment.appointmentId"),
        ("patientId",    "Appointment.patientId"),
        ("providerId",   "Appointment.providerId"),
        ("scheduledTime","Appointment.scheduledTime"),
        ("duration",     "Appointment.duration"),
        ("type",         "Appointment.type"),
        ("status",       "Appointment.status"),
    ]),
    "Diagnosis": ("diagnosis", [
        ("diagnosisId",  "Diagnosis.diagnosisId"),
        ("patientId",    "Diagnosis.patientId"),
        ("providerId",   "Diagnosis.providerId"),
        ("icdCode",      "Diagnosis.icdCode"),
        ("description",  "Diagnosis.description"),
        ("severity",     "Diagnosis.severity"),
        ("diagnosedDate","Diagnosis.diagnosedDate"),
    ]),
    "Prescription": ("prescription", [
        ("rxNumber",         "Prescription.rxNumber"),
        ("diagnosisId",      "Prescription.diagnosisId"),
        ("providerId",       "Prescription.providerId"),
        ("medication",       "Prescription.medication"),
        ("dosage",           "Prescription.dosage"),
        ("frequency",        "Prescription.frequency"),
        ("refillsRemaining", "Prescription.refillsRemaining"),
    ]),
}

data_bindings = {}  # entity_name -> data binding dict

for entity_name, (table_name, columns) in binding_map.items():
    binding_id = str(uuid.uuid4())
    property_bindings = []
    for col_name, prop_key in columns:
        property_bindings.append({
            "sourceColumnName": col_name,
            "targetPropertyId": prop_ids[prop_key],
        })

    data_bindings[entity_name] = {
        "id": binding_id,
        "dataBindingConfiguration": {
            "dataBindingType": "NonTimeSeries",
            "propertyBindings": property_bindings,
            "sourceTableProperties": {
                "sourceType": "LakehouseTable",
                "workspaceId": workspace_id,
                "itemId": lakehouse_id,
                "sourceTableName": table_name,
                "sourceSchema": "dbo",
            }
        }
    }

print("Data bindings built successfully.")
for name, db in data_bindings.items():
    cfg = db["dataBindingConfiguration"]
    print(f"  {name} -> table '{cfg['sourceTableProperties']['sourceTableName']}' ({len(cfg['propertyBindings'])} columns)")


## 7. Build Relationship Types and Contextualizations

Six relationships connect the five entity types. Each relationship includes a **contextualization** — the data binding that tells Fabric which table and columns to use for resolving the relationship at query time.

In [ ]:
# ─── RELATIONSHIP DEFINITIONS ─────────────────────────────────────────────────
# Each tuple: (name, source_entity, target_entity, bridge_table,
#               source_key_column, source_key_prop, target_key_column, target_key_prop)

relationship_specs = [
    ("has_appointment", "Patient",   "Appointment",  "appointment",
     "patientId",     "Patient.patientId",       "appointmentId", "Appointment.appointmentId"),

    ("sees",           "Provider",  "Appointment",  "appointment",
     "providerId",    "Provider.providerId",     "appointmentId", "Appointment.appointmentId"),

    ("diagnosed_with", "Patient",   "Diagnosis",    "diagnosis",
     "patientId",     "Patient.patientId",       "diagnosisId",   "Diagnosis.diagnosisId"),

    ("diagnoses",      "Provider",  "Diagnosis",    "diagnosis",
     "providerId",    "Provider.providerId",     "diagnosisId",   "Diagnosis.diagnosisId"),

    ("treated_by",     "Diagnosis", "Prescription", "prescription",
     "diagnosisId",   "Diagnosis.diagnosisId",   "rxNumber",      "Prescription.rxNumber"),

    ("prescribes",     "Provider",  "Prescription", "prescription",
     "providerId",    "Provider.providerId",     "rxNumber",      "Prescription.rxNumber"),
]

relationship_definitions = []
contextualization_definitions = []

for (rel_name, src_entity, tgt_entity, bridge_table,
     src_col, src_prop_key, tgt_col, tgt_prop_key) in relationship_specs:

    rel_id = next_id()

    rel_def = {
        "namespace": "usertypes",
        "id": rel_id,
        "name": rel_name,
        "namespaceType": "Custom",
        "source": {"entityTypeId": et_ids[src_entity]},
        "target": {"entityTypeId": et_ids[tgt_entity]},
    }
    relationship_definitions.append((rel_id, rel_def))

    ctx_id = str(uuid.uuid4())
    ctx_def = {
        "id": ctx_id,
        "dataBindingTable": {
            "sourceType": "LakehouseTable",
            "workspaceId": workspace_id,
            "itemId": lakehouse_id,
            "sourceTableName": bridge_table,
            "sourceSchema": "dbo",
        },
        "sourceKeyRefBindings": [
            {"sourceColumnName": src_col, "targetPropertyId": prop_ids[src_prop_key]}
        ],
        "targetKeyRefBindings": [
            {"sourceColumnName": tgt_col, "targetPropertyId": prop_ids[tgt_prop_key]}
        ],
    }
    contextualization_definitions.append((rel_id, ctx_id, ctx_def))

print("Relationship definitions built successfully.")
for rel_id, rel_def in relationship_definitions:
    print(f"  {rel_def['name']}: {rel_def['source']['entityTypeId']} -> {rel_def['target']['entityTypeId']}")


## 8. Assemble the Full Definition Payload

All the JSON structures are base64-encoded and assembled into the `definition.parts` array required by the Fabric Ontology REST API.

In [ ]:
# ─── ASSEMBLE DEFINITION PARTS ────────────────────────────────────────────────

parts = []

# 1. .platform file
platform = {
    "metadata": {
        "type": "Ontology",
        "displayName": ontology_display_name,
    }
}
parts.append({
    "path": ".platform",
    "payload": to_b64(platform),
    "payloadType": "InlineBase64",
})

# 2. definition.json (empty object per spec)
parts.append({
    "path": "definition.json",
    "payload": to_b64({}),
    "payloadType": "InlineBase64",
})

# 3. Entity Types + Data Bindings
for entity_name in ["Patient", "Provider", "Appointment", "Diagnosis", "Prescription"]:
    et_id = et_ids[entity_name]
    et_def = entity_definitions[entity_name]
    db_def = data_bindings[entity_name]

    # EntityType definition.json
    parts.append({
        "path": f"EntityTypes/{et_id}/definition.json",
        "payload": to_b64(et_def),
        "payloadType": "InlineBase64",
    })

    # DataBinding file
    parts.append({
        "path": f"EntityTypes/{et_id}/DataBindings/{db_def['id']}.json",
        "payload": to_b64(db_def),
        "payloadType": "InlineBase64",
    })

# 4. Relationship Types + Contextualizations
for rel_id, rel_def in relationship_definitions:
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/definition.json",
        "payload": to_b64(rel_def),
        "payloadType": "InlineBase64",
    })

for rel_id, ctx_id, ctx_def in contextualization_definitions:
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json",
        "payload": to_b64(ctx_def),
        "payloadType": "InlineBase64",
    })

print(f"Total definition parts: {len(parts)}")
print("\nPart paths:")
for p in parts:
    print(f"  {p['path']}")


## 9. Create the Ontology via REST API

This sends the `POST` request to the Fabric Ontology API. The notebook uses the built-in Fabric identity token for authentication — no manual credential setup required.

In [ ]:
# ─── GET AUTH TOKEN ───────────────────────────────────────────────────────────
# In a Fabric notebook, use the built-in token provider (notebookutils or mssparkutils).
from notebookutils import mssparkutils

access_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

# ─── BUILD REQUEST ───────────────────────────────────────────────────────────
url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/ontologies"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

body = {
    "displayName": ontology_display_name,
    "description": ontology_description,
    "definition": {
        "parts": parts,
    },
}

print(f"POST {url}")
print(f"Payload size: {len(json.dumps(body)):,} bytes")
print(f"Creating ontology '{ontology_display_name}'...")
print()

response = requests.post(url, headers=headers, json=body)

# ─── HANDLE RESPONSE ─────────────────────────────────────────────────────────
if response.status_code == 201:
    result = response.json()
    print("Ontology created successfully!")
    print(f"  ID:          {result.get('id')}")
    print(f"  Display Name:{result.get('displayName')}")
    print(f"  Workspace:   {result.get('workspaceId')}")
    print(f"  Type:        {result.get('type')}")

elif response.status_code == 202:
    # Long-running operation — poll until complete
    operation_id = response.headers.get("x-ms-operation-id")
    retry_after = int(response.headers.get("Retry-After", 30))
    print(f"Ontology creation accepted (async operation: {operation_id})")
    print(f"Polling every {retry_after}s for completion...")

    operation_url = f"https://api.fabric.microsoft.com/v1/operations/{operation_id}"
    for attempt in range(20):
        time.sleep(retry_after)
        poll = requests.get(operation_url, headers=headers)
        if poll.status_code == 200:
            status = poll.json().get("status", "")
            print(f"  Attempt {attempt+1}: {status}")
            if status.lower() in ("succeeded", "completed"):
                # Fetch the result
                result_url = f"{operation_url}/result"
                res = requests.get(result_url, headers=headers)
                if res.status_code == 200:
                    print("\nOntology created successfully!")
                    print(json.dumps(res.json(), indent=2))
                else:
                    print(f"\nOperation completed. Check your workspace for '{ontology_display_name}'.")
                break
            elif status.lower() == "failed":
                print("\nOperation FAILED.")
                print(json.dumps(poll.json(), indent=2))
                break
        else:
            print(f"  Poll returned {poll.status_code}")
    else:
        print("\nTimed out waiting for operation to complete. Check Fabric portal.")

else:
    print(f"ERROR {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)


## 10. Verify the Ontology

List all ontology items in the workspace to confirm creation.

In [ ]:
# List all items of type Ontology in the workspace
items_df = fabric.list_items(workspace=workspace_name, type="Ontology")
display(items_df)


## Summary

This notebook created the **HealthcareSystemOntology** with:

| Component | Count | Details |
|---|---|---|
| Entity Types | 5 | Patient, Provider, Appointment, Diagnosis, Prescription |
| Data Bindings | 5 | One per entity type, bound to Lakehouse tables |
| Entity Keys | 5 | patientId, providerId, appointmentId, diagnosisId, rxNumber |
| Relationship Types | 6 | has_appointment, sees, diagnosed_with, diagnoses, treated_by, prescribes |
| Contextualizations | 6 | One per relationship, using bridge tables |

**Next steps:**
- Open the ontology in the Fabric portal to see the visual canvas
- Use **Fabric IQ** to ask natural-language questions against this ontology
- Enrich the ontology with additional entity types or relationships as needed
